In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GraphConv
import torch_geometric.transforms as T
import numpy as np
from torch_geometric.loader import GraphSAINTRandomWalkSampler
from torch_geometric.typing import WITH_TORCH_SPARSE
from torch_geometric.utils import degree

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = dataset[0]
row, col = data.edge_index
data.edge_weight = 1. / degree(col, data.num_nodes)[col]

In [3]:
loader = GraphSAINTRandomWalkSampler(data, batch_size=6000, walk_length=2,
                                     num_steps=5, sample_coverage=100,
                                     save_dir=dataset.processed_dir,
                                     num_workers=4)

In [4]:
class GSAINT(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        in_channels = dataset.num_node_features
        out_channels = dataset.num_classes
        self.conv1 = GraphConv(in_channels, hidden_channels)
        self.conv2 = GraphConv(hidden_channels, hidden_channels)
        self.conv3 = GraphConv(hidden_channels, hidden_channels)
        self.lin = torch.nn.Linear(3 * hidden_channels, out_channels)

    def set_aggr(self, aggr):
        self.conv1.aggr = aggr
        self.conv2.aggr = aggr
        self.conv3.aggr = aggr

    def forward(self, x0, edge_index, edge_weight=None):
        x1 = F.relu(self.conv1(x0, edge_index, edge_weight))
        x1 = F.dropout(x1, p=0.2, training=self.training)
        x2 = F.relu(self.conv2(x1, edge_index, edge_weight))
        x2 = F.dropout(x2, p=0.2, training=self.training)
        x3 = F.relu(self.conv3(x2, edge_index, edge_weight))
        x3 = F.dropout(x3, p=0.2, training=self.training)
        x = torch.cat([x1, x2, x3], dim=-1)
        x = self.lin(x)
        return x.log_softmax(dim=-1)

model = GSAINT(hidden_channels=64).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [12]:
def train():
    model.train()
    model.set_aggr('add')

    total_loss = total_examples = 0
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()

        edge_weight = data.edge_norm * data.edge_weight
        out = model(data.x, data.edge_index, edge_weight)
        loss = F.nll_loss(out, data.y, reduction='none')
        loss = (loss * data.node_norm)[data.train_mask].sum()

        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_nodes
        total_examples += data.num_nodes
    return total_loss / total_examples


@torch.no_grad()
def test():
    model.eval()
    model.set_aggr('mean')

    out = model(data.x.to(device), data.edge_index.to(device))
    pred = out.argmax(dim=-1)
    correct = pred.eq(data.y.to(device))

    accs = []
    for _, mask in data('test_mask'):
        accs.append(correct[mask].sum().item() / mask.sum().item())
    return accs[0]

In [13]:
for epoch in range(0, 200):
    loss = train()
    acc = test()
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Accuracy: {acc:.4f}')

Epoch: 000, Loss: 0.0905, Accuracy: 0.4560
Epoch: 001, Loss: 0.0777, Accuracy: 0.4700
Epoch: 002, Loss: 0.0587, Accuracy: 0.5010
Epoch: 003, Loss: 0.0381, Accuracy: 0.5150
Epoch: 004, Loss: 0.0194, Accuracy: 0.5520
Epoch: 005, Loss: 0.0079, Accuracy: 0.5650
Epoch: 006, Loss: 0.0031, Accuracy: 0.5810
Epoch: 007, Loss: 0.0013, Accuracy: 0.6330
Epoch: 008, Loss: 0.0006, Accuracy: 0.6680
Epoch: 009, Loss: 0.0004, Accuracy: 0.7000
Epoch: 010, Loss: 0.0002, Accuracy: 0.7140
Epoch: 011, Loss: 0.0002, Accuracy: 0.7140
Epoch: 012, Loss: 0.0001, Accuracy: 0.7130
Epoch: 013, Loss: 0.0001, Accuracy: 0.7130
Epoch: 014, Loss: 0.0001, Accuracy: 0.7170
Epoch: 015, Loss: 0.0001, Accuracy: 0.7170
Epoch: 016, Loss: 0.0001, Accuracy: 0.7220
Epoch: 017, Loss: 0.0001, Accuracy: 0.7220
Epoch: 018, Loss: 0.0001, Accuracy: 0.7250
Epoch: 019, Loss: 0.0001, Accuracy: 0.7250
Epoch: 020, Loss: 0.0001, Accuracy: 0.7270
Epoch: 021, Loss: 0.0000, Accuracy: 0.7250
Epoch: 022, Loss: 0.0001, Accuracy: 0.7280
Epoch: 023,

In [14]:
torch.save(model.state_dict(), 'cora_gsaint.pt')